In [1]:
import shap
import spacetimeformer as stf
import sys
sys.path.append('../../bats_transformer')
from data.bats_dataset import *
from tqdm import tqdm
import numpy as np
import pandas as pd

In [2]:
dataset_path = "../../bats_transformer/data/2022_barn_daytime_2secs/splits"
dataset_path = "../../bats_transformer/data/2022_barn_2secs_myca/splits"

In [3]:
ignore_cols = ["FreqLedge","AmpK@end", "Fc", "FBak15dB  ", "FBak32dB", "EndF", "FBak20dB", "LowFreq", "Bndw20dB", 
               "CallsPerSec", "EndSlope", "SteepestSlope", "StartSlope", "Bndw15dB", "HiFtoUpprKnSlp", "HiFtoKnSlope", 
               "DominantSlope", "Bndw5dB", "PreFc500", "PreFc1000", "PreFc3000", "KneeToFcSlope", "TotalSlope", 
               "PreFc250", "CallDuration", "CummNmlzdSlp", "DurOf32dB", "SlopeAtFc", "LdgToFcSlp", "DurOf20dB", "DurOf15dB", 
               "TimeFromMaxToFc", "KnToFcDur", "HiFtoFcExpAmp", "AmpKurtosis", "LowestSlope", "KnToFcDmp", "HiFtoKnExpAmp", 
               "DurOf5dB", "KnToFcExpAmp", "RelPwr3rdTo1st", "LnExpB_StartAmp", "Filter", "HiFtoKnDmp", "LnExpB_EndAmp", 
               "HiFtoFcDmp", "AmpSkew", "LedgeDuration", "KneeToFcResidue", "PreFc3000Residue", "AmpGausR2", "PreFc1000Residue", 
               "Amp1stMean", "LdgToFcExp", "FcMinusEndF", "Amp4thMean", "HiFtoUpprKnExp", "HiFtoKnExp", "KnToFcExp", "UpprKnToKnExp", 
               "Kn-FcCurviness", "Amp2ndMean", "Quality", "HiFtoFcExp", "LnExpA_EndAmp", "RelPwr2ndTo1st", "LnExpA_StartAmp", 
               "HiFminusStartF", "Amp3rdMean", "PreFc500Residue", "Kn-FcCurvinessTrndSlp", "PreFc250Residue", "AmpVariance", "AmpMoment", 
               "meanKn-FcCurviness", "MinAccpQuality", "AmpEndLn60ExpC", "AmpStartLn60ExpC", "Preemphasis", "MaxSegLnght" ,"Max#CallsConsidered" ]
ignore_cols += ["Filename", "NextDirUp", 'Path', 'Version', 'Filter', 'Preemphasis', 'MaxSegLnght', "ParentDir", "file_id", "chirp_idx", "split"]

In [4]:
data_module = stf.data.DataModule(
    datasetCls = BatsCSVDataset,
    dataset_kwargs = {
        "root_path": dataset_path,
        "prefix": "split",
        "ignore_cols": ignore_cols,
        "time_col_name": "TimeIndex",
        "val_split": 0.05,
        "test_split": 0.05,
        "context_points": None,
        "target_points": 1,
        "random_seed": 31
    },
    batch_size=64,
    workers=4,
)

In [5]:
train_data = data_module.train_dataloader()
val_data = data_module.val_dataloader()
test_data = data_module.test_dataloader()

Trying to unpickle estimator QuantileTransformer from version 1.3.2 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
Trying to unpickle estimator QuantileTransformer from version 1.3.2 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
Trying to unpickle estimator QuantileTransformer from version 1.3.2 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations


In [6]:
truths = []
preds = []
errors = []

for batch in tqdm(test_data):
    x_t, x_c, y_t, y_c = batch
    mask = x_t > 0
    lengths = mask.sum(dim=1)
    feature_sums = x_c.sum(dim=1)
    for i, row in enumerate(feature_sums):
        pred = x_c[i, -1, :]
        truths.append(y_c[i].numpy()[0])
        preds.append(pred.numpy())
        errors.append((y_c[i] - pred).numpy()[0])

truths, preds, errors = np.array(truths), np.array(preds), np.array(errors)

100%|██████████| 171/171 [02:20<00:00,  1.22it/s]


In [7]:
target_columns = train_data.dataset.target_cols
pd.DataFrame(preds)
# target_columns

,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,0.772126,-0.933910,-0.762999,-1.163437,0.239718,0.903683,-0.359275,0.754203,-0.771179,-0.630872,...,2.173842,1.520694,1.385272,-0.654628,0.934827,-0.997817,-1.005365,-0.893995,-0.927817,0.632165
1,1.176890,0.607463,-0.075363,0.039965,0.171300,-0.661008,-0.277422,-0.257231,-0.076070,0.327191,...,-0.957454,-0.983952,-1.650119,0.344149,0.565766,-0.421594,0.054633,-0.620830,-0.146856,-0.622307
2,1.025830,1.123415,-0.038756,0.122043,0.387327,-1.056047,-0.064385,-0.745434,-0.034418,-0.396465,...,-1.139629,-1.188318,0.946860,-1.007648,0.021938,-0.474852,-0.249088,-0.685384,-0.407799,-0.949176
3,0.880888,0.316330,0.200859,0.423264,0.786570,-1.225025,-0.076631,-0.397462,0.202195,-0.401642,...,-0.992646,-1.553309,1.051357,-0.640132,0.134561,0.713637,0.245746,0.374684,0.085857,-1.580633
4,0.648105,0.461215,0.263870,0.372881,0.122204,-0.695561,-0.085365,-0.549718,0.262717,-0.407690,...,-1.180909,-1.032372,-0.211774,-0.324352,0.236443,0.349963,-0.032632,0.077610,-0.147364,-0.810809
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10890,-0.337506,1.851734,-0.060547,-0.215071,-0.299171,0.564425,0.063359,-0.500837,-0.059694,-0.306993,...,-0.782309,-0.171854,0.189488,-0.208022,0.861957,-0.124816,0.176455,-0.043341,0.299760,0.725263
10891,-0.593940,0.461215,0.966904,1.468552,2.497742,-2.213270,-0.264423,-0.040641,0.967545,1.359403,...,-0.297666,-0.918029,-1.914515,-0.702471,-1.855089,0.660126,1.153157,0.269839,0.630332,-2.027928
10892,-0.746389,0.684763,-0.281842,-0.215150,-1.349494,0.938610,0.272128,-1.362654,-0.289281,-0.159969,...,1.175872,0.325262,-1.915423,-2.230427,0.796607,-0.028828,-0.232959,-0.719716,-0.613898,0.853575
10893,1.035790,0.607463,-0.504513,-0.361046,0.717629,-1.799968,-1.179727,0.618865,-0.514725,-1.431378,...,-0.949547,1.687946,0.909459,-0.415832,-0.183503,-0.176951,-0.388032,-0.246807,-0.428584,-0.967221


In [8]:
mae = np.abs(errors).mean(axis=0)
mse = (errors * errors).mean(axis=0)

In [9]:
mse_df = pd.DataFrame(np.array([target_columns, mse]).T)
mse_df = mse_df.set_index(0)
# mse_df[1] = mse_df[1].round(6)
pd.Series(mse, index=target_columns)

TimeInFile          0.051738
PrecedingIntrvl     1.218906
HiFreq              1.873188
Bndwdth             1.925257
FreqMaxPwr          2.216121
PrcntMaxAmpDur      2.147612
FreqKnee            2.062285
PrcntKneeDur        1.690400
StartF              1.866646
UpprKnFreq          2.115920
HiFtoUpprKnAmp      2.051163
HiFtoKnAmp          1.907255
HiFtoFcAmp          1.904514
UpprKnToKnAmp      16.225233
KnToFcAmp           1.792927
LdgToFcAmp          2.048268
FreqCtr             1.753336
FFwd32dB            2.292477
FFwd20dB            2.022223
FFwd15dB            1.892968
FBak5dB             1.969780
FFwd5dB             1.936920
Bndw32dB            2.405529
Amp1stQrtl          2.082272
Amp2ndQrtl          2.045012
Amp3rdQrtl          1.754117
Amp4thQrtl          1.730879
1st10kHzSlp         2.220807
1st5to15kHzSlp      4.431187
1st10kHzExp         2.176147
1st5to15kHzExp      4.485902
AmpK@start          1.851575
dtype: float32

In [10]:
average_loss_per_row = mse_df.mean(axis=1)
average_loss_per_row_no_outlier = mse_df.drop("UpprKnToKnAmp", axis=0).mean(axis=1)
print(average_loss_per_row.mean(), average_loss_per_row_no_outlier.mean())

2.504642642625 2.062042953677419
